# 01 — Environment check (about 25 GPU-minutes)

**Run this before spending real GPU hours.** It turns every unverified piece of
the pipeline into a confirmed pass or a concrete error message.

**Settings (right-hand panel):**
- Accelerator: **GPU T4 x2** (not P100 — the T4 has tensor cores for fp16)
- Internet: **On** (needs a phone-verified account)
- Secrets: **`HF_TOKEN`** (Gemma is a gated model)
- Input: the **project dataset** (recommended), or set `GITHUB_URL` below

**Important:** do not add cells that load a model into this notebook. A model
loaded here stays on the GPU and starves the training runs, which is what made
every arm fail with "out of memory" in the first attempt.

## 1. Hardware — without touching the GPU from this kernel

In [ ]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap",
                          "--format=csv"], capture_output=True, text=True).stdout)
    cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.split()
except FileNotFoundError:
    raise SystemExit("No GPU in this session. Settings -> Accelerator -> GPU T4 x2, then restart.")
native_bf16 = bool(cap) and float(cap[0]) >= 8.0
print("native bf16:", native_bf16,
      "" if native_bf16 else " <- expected on T4/P100; the pipeline trains in fp16")
# torch.cuda.is_bf16_supported() says True on a T4 because bf16 can be
# EMULATED, slowly. The pipeline checks compute capability instead.

## 2. Get the project, and check it is complete

In [ ]:
import os, glob, shutil, subprocess

# ---- Where the project comes from (pick ONE) -------------------------------
# A. Kaggle Dataset — recommended. Upload offline-health-ai-assist.zip as a new
#    dataset (Kaggle unpacks it), then "Add Input" -> your dataset. Edit nothing.
# B. GitHub — set GITHUB_URL. Private repository: add a GITHUB_TOKEN secret.
GITHUB_URL = ""   # e.g. "https://github.com/<you>/offline-health-ai-assist.git"
# -----------------------------------------------------------------------------
WORK = "/kaggle/working/repo"


def find_root(d):
    """Project root = the folder containing src/train.py, at any depth. Handles
    uploads that gained an extra top-level folder."""
    hits = glob.glob(os.path.join(d, "**", "src", "train.py"), recursive=True)
    hits.sort(key=lambda p: p.count(os.sep))
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None


def fresh_copy(fill):
    """Replace the code in WORK but keep progress (runs/, results/) from an
    earlier run in this session."""
    keep = "/kaggle/working/_keep"
    shutil.rmtree(keep, ignore_errors=True)
    old = find_root(WORK) if os.path.isdir(WORK) else None
    if old:
        for d in ("runs", "results"):
            if os.path.isdir(os.path.join(old, d)):
                shutil.move(os.path.join(old, d), os.path.join(keep, d))
    shutil.rmtree(WORK, ignore_errors=True)
    fill()
    root = find_root(WORK)
    if root and os.path.isdir(keep):
        for d in os.listdir(keep):
            shutil.move(os.path.join(keep, d), os.path.join(root, d))
    return root


dataset_root = next((r for r in (find_root(d) for d in sorted(glob.glob("/kaggle/input/*"))) if r), None)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

if dataset_root:
    source = f"Kaggle dataset ({dataset_root})"
    root = fresh_copy(lambda: shutil.copytree(dataset_root, WORK))   # inputs are read-only
elif GITHUB_URL:
    source = "GitHub"
    url = GITHUB_URL.replace("https://", f"https://{token}@") if token else GITHUB_URL
    def _clone():
        r = subprocess.run(["git", "clone", "--depth", "1", url, WORK], capture_output=True, text=True)
        if r.returncode:
            raise SystemExit("git clone failed:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
        # never leave a token in .git/config: /kaggle/working is saved with the output
        subprocess.run(["git", "-C", WORK, "remote", "set-url", "origin", GITHUB_URL])
    root = fresh_copy(_clone)
else:
    raise SystemExit("No project found. Attach the project dataset (Add Input), or set GITHUB_URL.")

if not root:
    raise SystemExit(f"{source} contains no src/train.py — the upload is incomplete.")
os.chdir(root)
for f in ("run_smoke_test.sh", "mobile/sync_rules.sh"):
    if os.path.exists(f):
        os.chmod(f, 0o755)
print(f"source: {source}\nproject root: {root}\n")

# Completeness and integrity BEFORE anything else. --strict also rejects files
# that merely DIFFER from the manifest: this copy came from the project zip, so
# a difference means a broken copy, and a broken copy fails later in ways that
# point nowhere near the cause. One altered data file previously made all six
# training arms fail identically, seconds in, before any of them saw the GPU.
r = subprocess.run(["python3", "scripts/verify_repo.py", "--strict"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode:
    raise SystemExit("This copy of the project is incomplete or altered (list above). "
                     "Upload the full zip as a Kaggle Dataset — see docs/KAGGLE.md.")

## 3. Dependencies

In [ ]:
# Pinned to the exact versions the pipeline was tested with. torch is NOT
# reinstalled: Kaggle ships a CUDA build matched to its drivers.
!pip install -q "transformers==5.17.0" "trl==1.13.0" "peft==0.19.1" "accelerate==1.13.0" "bitsandbytes==0.50.2" datasets pyyaml safetensors

In [ ]:
# Versions as a FRESH interpreter sees them — which is what the training
# subprocesses use. (Versions imported earlier in this kernel can be stale.)
r = subprocess.run(["python3", "-c", """
import torch, transformers, trl, peft, accelerate, bitsandbytes
for m in (torch, transformers, trl, peft, accelerate, bitsandbytes):
    print(f"{m.__name__:<14}{m.__version__}")
print(f"{'CUDA':<14}{torch.cuda.is_available()}")
"""], capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])
print("Record these in the thesis methods.")

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("No HF_TOKEN secret — Gemma (a gated model) will fail to download.")
    print("Add the secret, and accept the Gemma licence on its model page with the same account.")

## 4. CPU-side checks — should match what passes on any machine

In [ ]:
!bash run_smoke_test.sh 2>&1 | tail -45

## 5. Choose a GPU with enough free memory

In [ ]:
# Pick the GPU with the most free memory, and refuse to start if something is
# already holding memory. Every arm in the first attempt failed with "CUDA out
# of memory" within seconds — the QLoRA arm could not get even 20 MB — which
# means the GPU was occupied BEFORE training began. The usual cause is a model
# loaded in a cell of this notebook: it stays on the GPU until the kernel
# restarts, and every training run then competes with it.
try:
    q = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free,memory.total",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit("No GPU in this session. Settings -> Accelerator -> GPU T4 x2, then restart.")
gpus = []
for line in q.stdout.strip().splitlines():
    i, name, used, free, total = (x.strip() for x in line.split(","))
    gpus.append({"i": int(i), "name": name, "used": int(used), "free": int(free), "total": int(total)})
print(f"{'GPU':<5}{'name':<12}{'used MiB':>10}{'free MiB':>10}")
for g in gpus:
    print(f"{g['i']:<5}{g['name']:<12}{g['used']:>10}{g['free']:>10}")

if not gpus:
    raise SystemExit("nvidia-smi reported no GPUs. Settings -> Accelerator -> GPU T4 x2.")
best = max(gpus, key=lambda g: g["free"])
if best["free"] < 12000:
    print("\nNot enough free GPU memory to train. Something is already using it —")
    print("most likely a model loaded in an earlier cell of this notebook.")
    print("Fix: Run -> Restart & clear cell outputs, then run the cells in order,")
    print("without adding cells that load a model.")
    raise SystemExit(f"GPU {best['i']} has only {best['free']} MiB free (need 12000).")

os.environ["CUDA_VISIBLE_DEVICES"] = str(best["i"])        # inherited by every subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"\ntraining will use GPU {best['i']} with {best['free']} MiB free")

## 5b. The training file, before six arms are spent failing on it

Every arm reads the same file. If it is wrong, all six fail identically a few
seconds in and the GPU is never the problem — so check it once, here, cheaply.

In [ ]:
import json, subprocess
DATA = "data/seed/train_sample.jsonl"
r = subprocess.run(["python3", "data/validate_dataset.py", DATA, "--kind", "train"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode:
    raise SystemExit(f"{DATA} does not satisfy the schema (above). Training would fail "
                     f"on every arm for this reason. Re-upload the project zip.")

rows = [json.loads(l) for l in open(DATA) if l.strip()]
print(f"{len(rows)} rows, fields: {sorted(rows[0])}")
print(f"first question: {rows[0]['question'][:90]}")
print(f"first answer:   {rows[0]['answer'][:90]}")

## 6. Every training arm, ten steps each

Ten optimizer steps on the seed data per arm, on the primary model. This proves
each arm *runs* on this GPU and shows its peak memory — it produces no usable
model. `--max-steps` must never be used for real results.

In [ ]:
import time, json, re
MODEL = "qwen05"
ARMS = ["lora", "qlora4", "qat4", "qat8", "qat_mixed", "fp16"]
SMOKE = "/kaggle/working/smoke"
results = {}

for arm in ARMS:
    t0 = time.time()
    r = subprocess.run(["python3", "src/train.py", "--model", MODEL, "--method", arm,
                        "--data", "data/seed/train_sample.jsonl", "--max-steps", "10",
                        "--out", f"{SMOKE}/{MODEL}_{arm}"], capture_output=True, text=True)
    out = r.stdout + "\n" + r.stderr
    open(f"{SMOKE}_{arm}.log", "w").write(out)
    peak = re.search(r"peak GPU memory: ([\d.]+) GB", out)
    results[arm] = {"ok": r.returncode == 0, "seconds": round(time.time() - t0),
                    "peak_gb": float(peak.group(1)) if peak else None}
    if r.returncode == 0:
        print(f"PASS  {arm:<10}{results[arm]['seconds']:>5}s   peak {results[arm]['peak_gb']} GB")
    else:
        # ALWAYS show the reason. An earlier version printed only lines matching
        # an error pattern, so a failure whose output did not match that pattern
        # printed nothing at all — including a process killed by a signal, which
        # leaves no traceback.
        rc = r.returncode
        why = f"exit {rc}"
        if rc < 0:
            why = (f"killed by signal {-rc}" +
                   ("  (out-of-memory killer: CPU RAM, not GPU)" if rc == -9 else ""))
        print(f"FAIL  {arm:<10}{results[arm]['seconds']:>5}s   {why}")
        tail = [l for l in out.splitlines() if l.strip()][-20:]
        for l in tail:
            print("      | " + l.rstrip()[:170])
        print(f"      full log: {SMOKE}_{arm}.log")
        print()

### If anything failed above

This prints one failing log in full. Paste it back — it is what identifies the
cause.

In [ ]:
failed = [a for a, r in results.items() if not r["ok"]]
if not failed:
    print("nothing failed")
else:
    print(f"failed: {', '.join(failed)}\nfull log for {failed[0]}:\n" + "=" * 70)
    print(open(f"{SMOKE}_{failed[0]}.log").read()[-6000:])

## 7. Are the saved checkpoints ordinary, loadable models?

In [ ]:
from safetensors import safe_open
for arm in ARMS:
    final = f"{SMOKE}/{MODEL}_{arm}/final"
    full, adapter = os.path.join(final, "model.safetensors"), os.path.join(final, "adapter_model.safetensors")
    if os.path.exists(full):
        with safe_open(full, "pt") as fh:
            leftover = [k for k in fh.keys() if "parametrizations" in k]
        print(f"{arm:<10} full model saved   fake-quant stripped: {not leftover}")
    elif os.path.exists(adapter):
        print(f"{arm:<10} adapter saved")
    else:
        print(f"{arm:<10} NOTHING SAVED")

## 8. The evaluation path, in deployed form, on this GPU

Scores two smoke checkpoints exactly as the sweep will: the LoRA adapter is
merged into the base, then the weights are rounded with llama.cpp's own Q4_0
arithmetic before generating. Five questions, a minute or two.

In [ ]:
gen_out = f"{SMOKE}/gen.jsonl"
if os.path.exists(gen_out):
    os.remove(gen_out)
for arm in ("lora", "qat4"):
    if not results.get(arm, {}).get("ok"):
        print(f"skip {arm}: training failed above")
        continue
    r = subprocess.run(["python3", "src/generate.py", "--run", f"{SMOKE}/{MODEL}_{arm}",
                        "--deploy", "q4_0", "--eval", "data/seed/eval_quality_sample.jsonl",
                        "--out", gen_out, "--append", "--batch-size", "8"],
                       capture_output=True, text=True)
    lines = [l for l in r.stdout.splitlines() if "simulated deployment" in l or "merged" in l]
    print(f"{'PASS' if r.returncode == 0 else 'FAIL'}  {arm}")
    for l in lines:
        print("      " + l.strip())
    if r.returncode:
        print("      " + "\n      ".join(r.stderr.strip().splitlines()[-5:]))
if os.path.exists(gen_out):
    rows = [json.loads(l) for l in open(gen_out)]
    print(f"\n{len(rows)} scored rows written (5 questions x 2 guard conditions per arm)")

## 9. Can every candidate model's tokenizer be loaded?

In [ ]:
import yaml
from transformers import AutoTokenizer
for c in yaml.safe_load(open("configs/models.yaml"))["candidates"]:
    if not c.get("hf_id") or c["hf_id"] == "TBD":
        continue
    try:
        tok = AutoTokenizer.from_pretrained(c["hf_id"])
        print(f"OK    {c['key']:<10} {c['hf_id']:<40} chat template: {bool(tok.chat_template)}")
    except Exception as e:
        print(f"FAIL  {c['key']:<10} {c['hf_id']:<40} {str(e)[:70]}")

## 10. Summary — paste this back, together with any FAIL lines above

In [ ]:
print(f"{'arm':<12}{'result':<8}{'time':>6}{'peak GPU':>11}")
print("-" * 37)
for arm, r in results.items():
    pk = f"{r['peak_gb']:.1f} GB" if r["peak_gb"] else "-"
    print(f"{arm:<12}{'PASS' if r['ok'] else 'FAIL':<8}{r['seconds']:>5}s{pk:>11}")
print(f"\n{sum(r['ok'] for r in results.values())}/{len(results)} arms run on this GPU")
json.dump(results, open("/kaggle/working/environment_check.json", "w"), indent=2)